# ..... Faster R-CNN Transfer Learning Object Detection on PASCAL VOC

This notebook demonstrates object detection on PASCAL VOC using Faster R-CNN with transfer learning (two-stage network).

The original implementation was tested on ..... and assisted by DeepSeek.  
For reference, see the original article:

.................................................................................


## 1. Download the PASCAL VOC dataset and inspect a few samples

First, we download PASCAL VOC 2012 and visualize several examples with ground-truth bounding boxes.


In [ ]:
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.datasets import VOCDetection
import torchvision.transforms as T
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import random
from tqdm import tqdm
import os

# Set seeds
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 1. Download VOC 2012
data_root = './data'
print("Checking/downloading PASCAL VOC 2012...")
voc_dataset = VOCDetection(root=data_root, year='2012', image_set='trainval', download=True)
print(f"Dataset ready. Total images: {len(voc_dataset)}")

# 2. VOC class names (20 + background)
VOC_CLASSES = [
    '__background__',   # index 0
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(VOC_CLASSES)}
NUM_CLASSES = len(VOC_CLASSES)   # 21

# 3. Convert VOC annotation to target dict
def voc_to_target(annotation):
    boxes = []
    labels = []
    for obj in annotation['annotation']['object']:
        bbox = obj['bndbox']
        boxes.append([float(bbox['xmin']), float(bbox['ymin']), float(bbox['xmax']), float(bbox['ymax'])])
        labels.append(CLASS_TO_IDX[obj['name']])
    return {
        'boxes': torch.tensor(boxes, dtype=torch.float32),
        'labels': torch.tensor(labels, dtype=torch.int64)
    }

# 4. Custom Dataset with transforms
class VOCDetectionDataset(torch.utils.data.Dataset):
    def __init__(self, voc_dataset, transforms=None):
        self.voc_dataset = voc_dataset
        self.transforms = transforms
    def __len__(self):
        return len(self.voc_dataset)
    def __getitem__(self, idx):
        image, annotation = self.voc_dataset[idx]
        image = T.ToTensor()(image)
        target = voc_to_target(annotation)
        if self.transforms:
            image, target = self.transforms(image, target)
        return image, target

class DetectionTransform:
    def __init__(self, size=(300, 300)):
        self.size = size
    def __call__(self, image, target):
        orig_h, orig_w = image.shape[1], image.shape[2]
        new_h, new_w = self.size
        image = T.Resize(self.size)(image)
        if len(target['boxes']) > 0:
            scale_x = new_w / orig_w
            scale_y = new_h / orig_h
            target['boxes'][:, 0] *= scale_x
            target['boxes'][:, 2] *= scale_x
            target['boxes'][:, 1] *= scale_y
            target['boxes'][:, 3] *= scale_y
        image = T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])(image)
        return image, target

# 5. Show random examples with ground-truth boxes (before training)
def draw_boxes(ax, boxes, labels, class_names, color='green', linewidth=2):
    for box, label in zip(boxes, labels):
        xmin, ymin, xmax, ymax = box
        rect = patches.Rectangle((xmin, ymin), xmax-xmin, ymax-ymin,
                                 linewidth=linewidth, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(xmin, ymin-5, class_names[label], fontsize=8,
                bbox=dict(facecolor=color, alpha=0.5), color='white')

def show_examples(dataset, num_examples=3, title="Ground Truth"):
    indices = random.sample(range(len(dataset)), num_examples)
    fig, axes = plt.subplots(1, num_examples, figsize=(15,5))
    if num_examples == 1:
        axes = [axes]
    for i, idx in enumerate(indices):
        img, target = dataset[idx]
        img_np = img.permute(1,2,0).cpu().numpy()
        mean = np.array([0.485,0.456,0.406]); std = np.array([0.229,0.224,0.225])
        img_np = np.clip(img_np * std + mean, 0, 1)
        axes[i].imshow(img_np)
        axes[i].axis('off')
        boxes = target['boxes'].cpu().numpy()
        labels = target['labels'].cpu().numpy()
        draw_boxes(axes[i], boxes, labels, VOC_CLASSES, color='green')
        axes[i].set_title(f"{title}\n{len(boxes)} objects")
    plt.tight_layout()
    plt.savefig(f'{title.lower().replace(" ", "_")}.png')
    plt.show(block=True)

vis_dataset = VOCDetectionDataset(voc_dataset, transforms=DetectionTransform(size=(300,300)))
print("\n--- Showing 3 random images with ground-truth bounding boxes ---")
show_examples(vis_dataset, num_examples=3, title="Ground Truth")


## 2. Prepare the dataset for training

In [ ]:
# 6. Data splits and loaders
dataset_size = len(voc_dataset)
indices = list(range(dataset_size))
split = int(0.2 * dataset_size)
np.random.shuffle(indices)
train_indices, val_indices = indices[split:], indices[:split]

train_dataset = VOCDetectionDataset(voc_dataset, transforms=DetectionTransform(size=(300,300)))
val_dataset = VOCDetectionDataset(voc_dataset, transforms=DetectionTransform(size=(300,300)))

def collate_fn(batch): return tuple(zip(*batch))
train_loader = DataLoader(train_dataset, batch_size=8, sampler=torch.utils.data.SubsetRandomSampler(train_indices),
                          collate_fn=collate_fn, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=8, sampler=torch.utils.data.SubsetRandomSampler(val_indices),
                        collate_fn=collate_fn, num_workers=4)

print(f"Training samples: {len(train_indices)}, Validation samples: {len(val_indices)}")


## 3. Define the model and hyperparameters

This notebook uses Faster R-CNN pretrained on COCO (80 categories). The backbone is ResNet-50-like, and the features are passed to a Region Proposal Network (RPN). After the RPN selects candidate regions, they are resized and sent to the classifier. This is a classic two-stage network.


In [ ]:
# 7. Model
def get_model(num_classes=NUM_CLASSES):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights='DEFAULT')
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

model = get_model().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# 8. Training functions
def train_one_epoch(model, loader, optimizer, device, epoch):
    model.train()
    total_loss = 0.0
    progress = tqdm(loader, desc=f"Epoch {epoch} [Train]")
    for images, targets in progress:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        losses = sum(loss_dict.values())
        total_loss += losses.item()
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        progress.set_postfix(loss=losses.item())
    return total_loss / len(loader)

def evaluate(model, loader, device):
    was_training = model.training
    model.train()   # to get losses
    total_loss = 0.0
    with torch.no_grad():
        for images, targets in tqdm(loader, desc="Validating"):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            losses = sum(loss_dict.values())
            total_loss += losses.item()
    if not was_training:
        model.eval()
    return total_loss / len(loader)


## 4. Train the model

In [ ]:
# 9. Train
num_epochs = 10
for epoch in range(1, num_epochs+1):
    train_loss = train_one_epoch(model, train_loader, optimizer, device, epoch)
    val_loss = evaluate(model, val_loader, device)
    lr_scheduler.step()
    print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")


## 5. Save the model

In [ ]:
# 10. Save and reload
torch.save(model.state_dict(), 'fasterrcnn_voc_state_dict.pt')
print("Model state_dict saved.")


## 6. Run inference with the trained model

In [ ]:
# Reload safely
loaded_model = get_model()
loaded_model.load_state_dict(torch.load('fasterrcnn_voc_state_dict.pt', map_location=device))
loaded_model = loaded_model.to(device)
loaded_model.eval()
print("Model reloaded from state_dict.")



# 11. Predict on 2 random images and show BOTH ground truth (green) and predicted (red) boxes
test_indices = random.sample(range(len(val_dataset)), 2)
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
if len(test_indices) == 1:
    axes = [axes]

print("\n--- Predicting on 2 random test samples (green = ground truth, red = prediction) ---")
for i, idx in enumerate(test_indices):
    image_tensor, target = val_dataset[idx]   # keep ground truth target
    with torch.no_grad():
        prediction = loaded_model([image_tensor.to(device)])[0]

    # Denormalize image for display
    img_np = image_tensor.permute(1, 2, 0).cpu().numpy()
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img_np = img_np * std + mean
    img_np = np.clip(img_np, 0, 1)

    axes[i].imshow(img_np)
    axes[i].axis('off')

    # Draw ground truth boxes (green)
    gt_boxes = target['boxes'].cpu().numpy()
    gt_labels = target['labels'].cpu().numpy()
    draw_boxes(axes[i], gt_boxes, gt_labels, VOC_CLASSES, color='green', linewidth=2)

    # Draw predicted boxes (red) with confidence > 0.5
    keep = prediction['scores'] > 0.5
    pred_boxes = prediction['boxes'][keep].cpu().numpy()
    pred_labels = prediction['labels'][keep].cpu().numpy()
    pred_scores = prediction['scores'][keep].cpu().numpy()
    for box, label, score in zip(pred_boxes, pred_labels, pred_scores):
        xmin, ymin, xmax, ymax = box
        rect = patches.Rectangle((xmin, ymin), xmax-xmin, ymax-ymin,
                                 linewidth=2, edgecolor='red', facecolor='none')
        axes[i].add_patch(rect)
        axes[i].text(xmin, ymin-5, f"{VOC_CLASSES[label]}: {score:.2f}",
                     fontsize=8, bbox=dict(facecolor='red', alpha=0.5), color='white')

    axes[i].set_title(f"Sample {idx}\nGT: {len(gt_boxes)} objects, Pred: {len(pred_boxes)}")

plt.tight_layout()
plt.savefig('predictions_with_boxes.png')
plt.show(block=True)
print("Predictions saved to 'predictions_with_boxes.png'")

# Also print details
for i, idx in enumerate(test_indices):
    print(f"\n--- Sample {idx} ---")
    image_tensor, target = val_dataset[idx]
    with torch.no_grad():
        pred = loaded_model([image_tensor.to(device)])[0]
    keep = pred['scores'] > 0.5
    print("Ground truth:")
    for box, label in zip(target['boxes'].cpu().numpy(), target['labels'].cpu().numpy()):
        print(f"  {VOC_CLASSES[label]}: ({box[0]:.0f},{box[1]:.0f},{box[2]:.0f},{box[3]:.0f})")
    print("Predictions (score > 0.5):")
    for box, label, score in zip(pred['boxes'][keep].cpu().numpy(),
                                 pred['labels'][keep].cpu().numpy(),
                                 pred['scores'][keep].cpu().numpy()):
        print(f"  {VOC_CLASSES[label]}: {score:.2f} at ({box[0]:.0f},{box[1]:.0f},{box[2]:.0f},{box[3]:.0f})")

print("\n--- Training and inference completed successfully! ---")


The trained model performs object detection successfully.

This completes a simple object detection pipeline based on Faster R-CNN pretrained transfer learning.


## Contact

For job opportunities or HR inquiries, or for project collaboration: **yucongcai_business@outlook.com**  
For research-related inquiries: **yucongcai_research@outlook.com**


---

## Version log

| Version | Date | Change |
|---|---|---|
| v1.0 | 2026-08-03 | Initial rebuild from `assets/previous-resources/` (archive kept untouched). |

### v1.0 changes (2026-08-03)

| Change |
|---|
| No code changes needed (clean notebook) |